## Initial Replication of Estimates from [1] 

[1] Combined Postmenopausal Hormone Therapy and Cardiovascular Disease: Toward Resolving the Discrepancy between Observational Studies and the Women’s Health Initiative Clinical Trial, Prentice et al., 2005

In [1]:
import pandas as pd 
import numpy as np 
import os 
import sys 
from tqdm import tqdm
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from scipy.stats import zscore

In [2]:

# read tables
dir_path = '/Users//Documents/research/benchmarking-os/'
out   = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/outc_adj_bio.csv'))
ct_fu = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/adh_ht_pub.csv'))[['ID', 'ADHRATE', 'ENDDY', 'STARTDY', 'LOST', 'STOPHRT']] 
std_trt = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/dem_ctos_bio.csv'))[['ID', 'HRTARM', 'OSFLAG']]



In [3]:
# List of outcomes     
glbl_list = ['CHD', 'BREAST', 'STROKE', 'PE', 'ENDMTRL', 'COLORECTAL', 'BKHIP', 'DEATH']    
other_list = ['PTCA', 'DVT']

In [ ]:
# Get end of follow-up for CT patients 
# BTW, do we have to consider START-DAY? what about LOST for censoring?

# keep only those with ADHRATE not missing, and group by ID to get max ENDDY
# keep columns 'ID', 'ENDDY', and 'LOST'
# rename ENDDY to END_DY
# ct_end = ct_fu[ct_fu['ADHRATE'].notna()].groupby('ID')['ENDDY'].max().reset_index() 
# ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
# ct_end

# ct_end = ct_fu[ct_fu['ADHRATE'].notna()][['ADHRATE','ID', 'ENDDY', 'LOST']].rename(columns={'ENDDY': 'END_DY'})
# ct_end = ct_fu[['ADHRATE','ID', 'ENDDY', 'LOST']].rename(columns={'ENDDY': 'END_DY'})
# ct_end = ct_end.query('ADHRATE != 0.')[['ID','END_DY','LOST']]
ct_end = ct_fu[ct_fu['ADHRATE'].notna()].groupby('ID')['ENDDY'].max().reset_index() 
ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
ct_end

In [ ]:
ct_df = std_trt.drop_duplicates('ID')
ct_df = ct_df[ct_df['HRTARM'].isin(['E+P intervention', 'E+P control'])]
ct_df = ct_df.merge(ct_end, on='ID', how='left')
ct_df = ct_df.merge(out, on='ID', how='left')

# code variables HRTARM and OS 
ct_df['OS'] = 0 
ct_df['HRTARM'] = ct_df['HRTARM'].map({'E+P intervention': 1, 'E+P control': 0})

# print out first 10 rows
print(ct_df.shape)
print(ct_df[ct_df['HRTARM'] == 1].shape)
print(ct_df[ct_df['HRTARM'] == 0].shape)
ct_df.head(n=10)


In [6]:
# process outcomes 
for i in glbl_list + other_list: 
    ct_df[i+'_E']  = ((ct_df[i] == 1) & (ct_df[i+'DY'] <= ct_df['END_DY'])).astype(int)
    ct_df[i+'_DY'] = np.where(ct_df[i+'_E'] == 1, ct_df[i+'DY'], ct_df['END_DY'])
    ct_df[i+'_EDY'] = np.where(ct_df[i+'_E'] == 1, ct_df[i+'DY'], np.nan) 

# Global index
ct_df['GLBL_E'] = (ct_df[[j+'_E' for j in glbl_list]].sum(axis=1) > 0).astype(int)
ct_df['GLBL_DY'] = np.where(ct_df['GLBL_E'] == 1,
                            ct_df[[j+'_EDY' for j in glbl_list]].min(axis=1),
                            ct_df[[j+'_DY' for j in glbl_list]].min(axis=1))

# Select needed columns
ct_df = ct_df[['ID', 'OS', 'HRTARM'] + 
                [j+'_E' for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_DY' for j in glbl_list + other_list + ['GLBL']]]


In [ ]:
ct_df.query('HRTARM == 0 & CHD_E == 1')

In [ ]:
dir_path = '/Users//Documents/research/benchmarking-os/'
hyst    = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f2_ctos_bio.csv'))[['ID','HYST']]
pre_hrt  = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f43_ctos_bio.csv'))[['ID', 'TOTESTAT','TOTPSTAT']]
post_hrt = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f48_av1_os_pub.csv'))[['ID','ELSTYR','PLSTYR','HRTCMBP']]

In [ ]:
out.columns

In [ ]:
os_df_temp = std_trt.drop_duplicates('ID')
os_df_temp = os_df_temp[os_df_temp['OSFLAG'] == 'Yes']
os_df_temp = os_df_temp.merge(hyst, on='ID', how='left')
os_df_temp = os_df_temp.merge(pre_hrt, on='ID', how='left')
# os_df_temp = os_df_temp[os_df_temp['HYST'] == 'Yes']
print(os_df_temp.columns)
os_df_temp = os_df_temp.query('HYST.astype("string") == "Yes" | TOTESTAT.astype("string") == "Current user"')

# os_df_temp = os_df_temp[os_df_temp['TOTESTAT'] == 'Current user']
unc_hf = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/unc_hf_bio.csv'))[['ID', 'CHDYRHX', 'CHDEVERHX']]
os_df_temp = os_df_temp.merge(unc_hf, on='ID', how='left') 
print(os_df_temp['CHDYRHX'].value_counts())
print(os_df_temp['CHDEVERHX'].value_counts())
print(os_df_temp['TOTPSTAT'].value_counts())

In [ ]:
# construct os_df 
os_df = std_trt.drop_duplicates('ID')
os_df = os_df[os_df['OSFLAG'] == 'Yes']
os_df = os_df.merge(hyst, on='ID', how='left')
os_df = os_df[os_df['HYST'] == 'No']
os_df = os_df.merge(pre_hrt, on='ID', how='left')
print(os_df['TOTESTAT'].value_counts())
os_df = os_df[os_df['TOTESTAT'].isin(['Never used', 'Past user'])]
os_df = os_df.merge(post_hrt, on='ID', how='left')
os_df = os_df.merge(out, on='ID', how='left')

# 35551 (control) + 17503 (intervention) = 53054
print(os_df[os_df['TOTPSTAT'].isin(['Current user'])].shape)
print(os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user'])].shape)
os_df = os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user','Current user'])]
os_df['OS'] = 1
os_df['HRTARM'] = os_df['TOTPSTAT'].map({'Current user': 1, 'Never used': 0, 'Past user': 0})

# os_end_day = None
os_end_day = 6*365
os_df['END_DY'] = os_end_day if os_end_day is not None else os_df['ENDFOLLOWDY']
# os_df['END_DY'] = os_df.apply(lambda x: x['DEATHDY'] if x['DEATHDY'] < os_end_day else os_end_day, axis=1)

# Process outcomes (same as CT)
for i in glbl_list + other_list:
    os_df[i+'_E'] = ((os_df[i] == 1) & (os_df[i+'DY'] <= os_df['END_DY'])).astype(int)
    os_df[i+'_DY'] = np.where(os_df[i+'_E'] == 1, os_df[i+'DY'], os_df['END_DY'])
    os_df[i+'_EDY'] = np.where(os_df[i+'_E'] == 1, os_df[i+'_DY'], np.nan)

# Global index
os_df['GLBL_E'] = (os_df[[j+'_E' for j in glbl_list]].sum(axis=1) > 0).astype(int)
os_df['GLBL_DY'] = np.where(os_df['GLBL_E'] == 1,
                            os_df[[j+'_EDY' for j in glbl_list]].min(axis=1),
                            os_df[[j+'_DY' for j in glbl_list]].min(axis=1))

# Select needed columns
os_df = os_df[['ID', 'OS', 'HRTARM'] + 
                [j+'_E' for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_DY' for j in glbl_list + other_list + ['GLBL']]]

os_df


# Below is what you would do for proper trial emulation LOL
# conditions = [
#     (((os_df['ELSTYR'] == 'Yes') & (os_df['PLSTYR'] == 'Yes')) | (os_df['HRTCMBP'] == 'Yes')),
#     ((os_df['ELSTYR'] == 'No') & (os_df['PLSTYR'] == 'No')),
#     (((os_df['ELSTYR'] == 'Yes') & (os_df['PLSTYR'] == 'No')) | ((os_df['ELSTYR'] == 'No') & (os_df['PLSTYR'] == 'Yes')))
# ]
# choices = [1, 0, -1]
# os_df['HRTGRP'] = np.select(conditions, choices, default=-2)
# os_df = os_df[os_df['HRTGRP'] != -2]
# os_df['HRTARM'] = (os_df['HRTGRP'] == 1).astype(int)


In [ ]:
# check outcome of breast cancer and CHD in both ct and os
print('Clinical Trial')
print(f'No. of women in control arm: {ct_df[ct_df["HRTARM"] == 0].shape[0]}')
print(f'No. of women in intervention arm: {ct_df[ct_df["HRTARM"] == 1].shape[0]}')
print(f'No. of CHD events in control arm: {ct_df.query("HRTARM == 0 & CHD_E == 1").shape[0]}')
print(f'No. of CHD events in intervention arm: {ct_df.query("HRTARM == 1 & CHD_E == 1").shape[0]}')
print(f'No. of breast cancer events in control arm: {ct_df.query("HRTARM == 0 & BREAST_E == 1").shape[0]}')
print(f'No. of breast cancer events in intervention arm: {ct_df.query("HRTARM == 1 & BREAST_E == 1").shape[0]}')
print(f'No. of stroke events in control arm: {ct_df.query("HRTARM == 0 & STROKE_E == 1").shape[0]}')
print(f'No. of stroke events in intervention arm: {ct_df.query("HRTARM == 1 & STROKE_E == 1").shape[0]}')
print()
print()

print('Observational Study')
print(f'No. of women in control arm: {os_df[os_df["HRTARM"] == 0].shape[0]}')
print(f'No. of women in intervention arm: {os_df[os_df["HRTARM"] == 1].shape[0]}')
print(f'No. of CHD events in control arm: {os_df.query("HRTARM == 0 & CHD_E == 1").shape[0]}')
print(f'No. of CHD events in intervention arm: {os_df.query("HRTARM == 1 & CHD_E == 1").shape[0]}')
print(f'No. of breast cancer events in control arm: {os_df.query("HRTARM == 0 & BREAST_E == 1").shape[0]}')
print(f'No. of breast cancer events in intervention arm: {os_df.query("HRTARM == 1 & BREAST_E == 1").shape[0]}')
print(f'No. of stroke events in control arm: {os_df.query("HRTARM == 0 & STROKE_E == 1").shape[0]}')
print(f'No. of stroke events in intervention arm: {os_df.query("HRTARM == 1 & STROKE_E == 1").shape[0]}')


In [12]:
# Combine CT and OS data
ctos_df = pd.concat([ct_df, os_df], ignore_index=True)


In [ ]:
ctos_df

# Analysis

In [ ]:
import pandas.api.types as ptypes

ctos_temp = ctos_df.copy()
# Dictionary to specify which features are categorical
categorical_features = {
    'dem_ctos_bio.csv': {'AGE': False, 'ETHNIC': True, 'EDUC': True}, 
    'f80_ctos_bio.csv': {'BMI': False}, 
    'f34_ctos_bio.csv': {'SMOKING': True}, 
    'f31_ctos_bio.csv': {'MENO': False}, 
    'f151_ctos_bio.csv': {'PHYSFUN': False}    
}

new_feature_dict = { 
    'dem_ctos_bio.csv': ['AGE', 'ETHNIC_White', \
                         'EDUC_Some post-graduate or professional', \
                         'EDUC_Some college or Associate Degree'],
    'f80_ctos_bio.csv': ['BMI'],
    'f34_ctos_bio.csv': ['SMOKING_Past Smoker', 'SMOKING_Current Smoker'],
    'f31_ctos_bio.csv': ['MENO'],
    'f151_ctos_bio.csv': ['PHYSFUN']
}

# dfs = []  # Store all dataframes to concatenate later
new_dir_path = dir_path + 'whi/data/data/main_study/csv'

for filename, f_dict in categorical_features.items():
    # Read the data
    df = pd.read_csv(os.path.join(new_dir_path, filename))
    if filename == 'f80_ctos_bio.csv': 
        df = df.query('F80VTYP == "Screening"')
    elif filename == 'f151_ctos_bio.csv': 
        idx = df.groupby('ID')['F151DAYS'].idxmin().reset_index(drop=True)
        df = df.loc[idx, :].reset_index(drop=True)[['ID','PHYSFUN']]
    # Select needed columns
    features = list(f_dict.keys())
    df = df[['ID'] + features]
    
    # Separate ID column
    id_col = df['ID']
    print(f"Processed {filename}")
    print(df.shape)

    orig_cols = ctos_temp.columns.tolist()
    ctos_temp = ctos_temp.merge(df, on='ID', how='left')

    # Handle continuous and categorical features separately
    cont_features = [f for f in features if not f_dict[f]]
    cat_features = [f for f in features if f_dict[f]]
    
    # Handle continuous features
    if cont_features:
        cont_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
        ctos_temp[cont_features] = cont_imputer.fit_transform(ctos_temp[cont_features])
    
    # Handle categorical features
    if cat_features:
        cat_imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        ctos_temp[cat_features] = cat_imputer.fit_transform(ctos_temp[cat_features])
        
        # One-hot encode categorical features
        ctos_temp = pd.get_dummies(ctos_temp, columns=cat_features, prefix=cat_features)

    if filename == 'dem_ctos_bio.csv': 
        ctos_temp = ctos_temp.rename(columns={'ETHNIC_White (not of Hispanic origin)': 'ETHNIC_White'})

    ctos_temp = ctos_temp[orig_cols + new_feature_dict[filename]]

ctos_temp = ctos_temp.astype({col: int for col in ctos_temp.select_dtypes(include='bool').columns})
display(ctos_temp)    


In [ ]:
# hazard ratios for stroke, breast cancer, and CHD in clinical trial vs observational study 

## CT 
ct_df = ctos_temp.query('OS == 0')
ct_df_sub = ct_df[['ID','HRTARM', 'STROKE_E', 'BREAST_E', 'CHD_E','STROKE_DY', 'BREAST_DY', 'CHD_DY']]
ct_df_chd = ct_df[['HRTARM', 'CHD_E', 'CHD_DY']]
ct_df_chd = ct_df_chd[ct_df_chd['CHD_DY'].notna()]

ct_df_stroke = ct_df[['HRTARM', 'STROKE_E', 'STROKE_DY']]
ct_df_stroke = ct_df_stroke[ct_df_stroke['STROKE_DY'].notna()]

ct_df_breast = ct_df[['HRTARM', 'BREAST_E', 'BREAST_DY']]
ct_df_breast = ct_df_breast[ct_df_breast['BREAST_DY'].notna()]

from lifelines import CoxPHFitter

def get_hr(df, Y, E, event_name, HR_cov='HRTARM', study_type='Clinical Trial'): 
    cph = CoxPHFitter()
    cph.fit(df, duration_col=Y, event_col=E)
    cph.print_summary()
    cHR = cph.hazard_ratios_[HR_cov]
    cis = cph.confidence_intervals_
    lower = np.exp(cis['95% lower-bound'][HR_cov])
    upper = np.exp(cis['95% upper-bound'][HR_cov])
    print(f'Hazard ratio for {event_name} in {study_type}: {np.round(cHR, 2)} (95% CI: {np.round(lower, 2)}, {np.round(upper, 2)})')

get_hr(ct_df_chd, 'CHD_DY', 'CHD_E', 'CHD')
get_hr(ct_df_stroke, 'STROKE_DY', 'STROKE_E', 'Stroke')
get_hr(ct_df_breast, 'BREAST_DY', 'BREAST_E', 'Breast Cancer')


In [ ]:
# OS 
os_df = ctos_temp.query('OS == 1')
features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

os_df_sub = os_df[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]

get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')




# Fix with including the confounder (doesn't work)

In [17]:
os_df = ctos_temp.query('OS == 1')
pre_hrt_time  = pd.read_csv(os.path.join(dir_path, \
            'whi/data/data/main_study/csv/f43_ctos_bio.csv'))[['ID', 'TOTPTIME']]
os_df = os_df.merge(pre_hrt_time, on='ID', how='left') 
os_df['TAU'] = os_df.apply(lambda x: x['TOTPTIME'] if x['HRTARM'] == 1 else 0, axis=1)
# remove TOTPTIME as column from os_df inplace 
os_df.drop(columns=['TOTPTIME'], inplace=True)




In [ ]:
features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN', 'TAU']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

os_df_sub = os_df[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]

get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

In [ ]:
ctr_df = os_df.query('HRTARM == 0')
trt_df = os_df.query('HRTARM == 1')
trt_df_l2 = trt_df.query('TAU < 2')
trt_df_l2_5 = trt_df.query('(TAU >= 2 & TAU < 5)')
trt_df_g5 = trt_df.query('TAU >= 5')
 
# randomly split the control into 3 groups 
n = ctr_df.shape[0]
idxs = np.arange(n)
np.random.shuffle(idxs)
idxs_groups = np.array_split(idxs, 3)
crt_df_1 = ctr_df.loc[idxs_groups[0], :]
crt_df_2 = ctr_df.loc[idxs_groups[1], :]
crt_df_3 = ctr_df.loc[idxs_groups[2], :]

#concatenate trt and ctrl groups 
os_df_l2 = pd.concat([trt_df_l2, crt_df_1], ignore_index=True)
os_df_l2_5 = pd.concat([trt_df_l2_5, crt_df_2], ignore_index=True)
os_df_g5 = pd.concat([trt_df_g5, crt_df_3], ignore_index=True)

features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

print(os_df.query('CHD_E == 1 & HRTARM == 0').shape)

os_df_sub = os_df_l2[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
print(os_df_sub.query('CHD_E == 1 & HRTARM == 0').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

os_df_sub = os_df_l2_5[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
# print(os_df_sub.query('CHD_E == 1 & HRTARM == 0').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

os_df_sub = os_df_g5[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
print(os_df_sub.query('CHD_E == 1 & HRTARM == 1').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

In [ ]:
# instead, try stratifying by TAU
os_df_l2 = os_df.query('TAU < 2')
os_df_l2_5 = os_df.query('(TAU >= 2 & TAU < 5) | HRTARM == 0') 
os_df_g5 = os_df.query('TAU >= 5 | HRTARM == 0')

features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

print(os_df.query('CHD_E == 1 & HRTARM == 0').shape)

os_df_sub = os_df_l2[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
print(os_df_sub.query('CHD_E == 1 & HRTARM == 0').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

os_df_sub = os_df_l2_5[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
# print(os_df_sub.query('CHD_E == 1 & HRTARM == 0').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

os_df_sub = os_df_g5[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
print(os_df_sub.query('CHD_E == 1 & HRTARM == 1').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')